# LLaVA-S - Domain Finetuning on CheXpert Plus

Adapts base LLaVA-1.5-7B to chest radiography by supervised finetuning on paired chest X-rays
and radiology reports, producing the model referred to as LLaVA-S. The notebook also contains a
first evaluation of the resulting model on the entity-probing benchmark.

## Inputs
- `chexpert_cache/` (image shards, manifest, report table)
- `test_vqa.jsonl` - used only to exclude its patients from training and, at the end, to evaluate
- Base model weights, downloaded automatically

## Outputs
- A LoRA adapter for LLaVA-1.5-7B (the LLaVA-S model)
- Entity-probing evaluation results for that model

## Why this stage exists
Base LLaVA has never seen a chest X-ray, so it describes radiographs in general visual language
and invents findings. This stage supplies the missing domain knowledge. It does not address the
model's single-image limitation, which the V-RAG finetuning stage handles.

## Environment check

Verifies the GPU, driver and available memory before anything is downloaded, so an unsuitable
runtime fails immediately rather than part-way through training.

In [ ]:
# ============================================================
#  Cell 0 — Environment gate.  READ THE OUTPUT BEFORE CONTINUING.
# ============================================================
#  This notebook needs torch >= 2.7. That is not a preference:
#    * torchao references torch.int1        -> needs torch >= 2.6
#    * torchao calls torch.utils._pytree.register_constant -> needs torch >= 2.7
#  An older template (e.g. RunPod's PyTorch 2.4.1) CANNOT run current unsloth,
#  and no amount of pinning fixes it. If this cell fails, change the template —
#  do not start patching.
#
#  Deliberately no `import torch` at module level before the install cell: an
#  imported torch would stay cached in this kernel even if pip replaced it.
import subprocess, sys, os

print("GPU:")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                      "--format=csv"], capture_output=True, text=True).stdout)

r = subprocess.run([sys.executable, "-c",
    "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"],
    capture_output=True, text=True)
print("torch:", r.stdout.strip() or r.stderr.strip())

if r.returncode == 0 and r.stdout.strip():
    _v = r.stdout.split()[0].split("+")[0]
    _t = tuple(int(x) for x in _v.split(".")[:2])
    if _t < (2, 7):
        print(f"\ntorch {_v} is too old. Current unsloth/torchao need >= 2.7.")
        print("   Take a pod on a PyTorch 2.7/2.8 + CUDA 12.8 template. Do not patch this.")
    else:
        print(f"\ntorch {_v} — good")

os.makedirs("/workspace/data/test_vqa", exist_ok=True)
os.makedirs("/workspace/vqa", exist_ok=True)
print("/workspace ready")
print("\nUpload test_vqa.jsonl to /workspace/data/test_vqa/ — it is the one file")
print("   the Kaggle bundle cannot supply (you generated it yourself).")


## Configuration

All training settings: base model, LoRA rank and target modules, batch size, gradient
accumulation, learning rate, sequence length, and the number of samples to train on. Paths for
the cache and outputs are also defined here.

In [ ]:
# ============================================================
#  ALL CONFIGURABLE CONSTANTS — edit only here
# ============================================================
import os
from pathlib import Path

SAMPLE_SIZE      = 100000    # paper uses 100,098. Any value <= ~190k works: every
                             # frontal image is cached, so changing this costs nothing.
SEED             = 42
MIN_REPORT_LEN   = 50        # characters; discard junk/empty reports

# LoRA — matches the paper
LORA_R           = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05

# Training — matches the paper
LR               = 5e-5
EPOCHS           = 1
LR_SCHEDULER     = "cosine"
WARMUP_RATIO     = 0.03
MAX_SEQ_LEN      = 2048      # a cap, not a pad target. ~830 tokens/sample actual
                             # (576 image + ~210 report + template), so nothing truncates.

# RTX 3090 (24 GB): 4-bit 7B leaves headroom, so do more work per step instead of
# accumulating. Effective batch stays 4 — same recipe as the paper.
BATCH_SIZE       = 2         # drop to 1 (and GRAD_ACCUM to 4) if you OOM
GRAD_ACCUM       = 2

# Paths
WORK_DIR         = "/workspace"
VQA_SRC          = f"{WORK_DIR}/data/test_vqa"
VQA_DST          = f"{WORK_DIR}/vqa/test_vqa_dataset"
DATASET_JSONL    = f"{WORK_DIR}/vqa/llava_s_dataset.jsonl"
DATASET_PARQUET  = f"{WORK_DIR}/vqa/llava_s_dataset_meta.parquet"
ADAPTER_SAVE_DIR = f"{WORK_DIR}/vqa/llava_s_chexpert"

# Local cache (built once by kaggle-chexpert-cache-builder.ipynb)
PNG_CACHE_DIR    = Path(f"{WORK_DIR}/png_cache")            # 336x336 grayscale, ~11.6 GB
LOCAL_PARQUET    = Path(f"{WORK_DIR}/chexpert_plus_full.parquet")
IMG_SIZE         = 336       # LLaVA-1.5 CLIP ViT-L/14-336 crop. Do not change.

# Model
BASE_MODEL       = "llava-hf/llava-1.5-7b-hf"   # 14.1 GB download.
                                                # "unsloth/llava-1.5-7b-hf-bnb-4bit" is the
                                                # same weights pre-quantised: 4.0 GB, faster
                                                # load, lower RAM. Swap if disk/RAM is tight.
LOAD_IN_4BIT     = True

# Dataloader — images come off local disk, so this is cheap. Don't exceed the box.
DATALOADER_WORKERS = min(4, max(1, (os.cpu_count() or 2) - 2))
PREFETCH_WORKERS   = 24      # only used if the cache is incomplete

# Redivis — fallback only. With the cache present these are never touched.
DATASET_REF   = "chexpert_plus:5yyj"
PNG_TRAIN_REF = "PNG_train:s6cj"
REPORT_TABLE  = "df_chexpert_plus_240401:bavj"

print("Constants set")
print(f"   SAMPLE_SIZE        : {SAMPLE_SIZE:,}")
print(f"   batch              : {BATCH_SIZE} x {GRAD_ACCUM} accum = {BATCH_SIZE*GRAD_ACCUM} effective")
print(f"   steps              : ~{SAMPLE_SIZE // (BATCH_SIZE*GRAD_ACCUM):,}")
print(f"   cpu_count          : {os.cpu_count()}  -> DATALOADER_WORKERS = {DATALOADER_WORKERS}")
print(f"   cache              : {PNG_CACHE_DIR}")


## Install

Installs the training stack into an isolated environment: the quantisation, LoRA and trainer
libraries pinned to compatible versions.

In [ ]:
# ============================================================
#  Install — only if needed
# ============================================================
#  On a PyTorch 2.7/2.8 + CUDA 12.8 template this is a plain `pip install unsloth`.
#  No constraints file, no torch pin, no typing_extensions fix, no torchvision
#  juggling — every one of those workarounds existed solely because an older
#  template shipped a torch that current unsloth cannot run. On a modern base they
#  are not needed and actively cause harm (a stale pin can drag torch backwards).
#
#  Set SKIP_INSTALL = True once everything is in place, so Run All can't re-resolve
#  and disturb a working environment.
SKIP_INSTALL = False

import subprocess, sys, importlib.metadata as _m

def sh(cmd):
    print(f"$ {cmd}", flush=True)
    rc = subprocess.run(cmd, shell=True).returncode
    if rc:
        raise RuntimeError(f"install failed (exit {rc}): {cmd}")

def _have(pkg):
    try:    return _m.version(pkg)
    except Exception: return None

if SKIP_INSTALL:
    print("SKIP_INSTALL=True — verifying only")
else:
    if _have("unsloth"):
        print(f"unsloth {_have('unsloth')} already present — skipping install")
    else:
        sh("pip install unsloth")
    sh("pip install -q redivis Pillow tqdm datasets transformers trl peft "
       "bitsandbytes pyarrow pandas kaggle matplotlib")

# ---- verify in a FRESH process ----
# torchvision.ops forces the C++ extension to load: that is what actually breaks on
# a torch/torchvision mismatch, so a bare import proves nothing.
check = subprocess.run([sys.executable, "-c",
    "import torch, torchvision, torchvision.ops, torchao, unsloth, importlib.metadata as m;"
    "print('torch       :', torch.__version__);"
    "print('torchvision :', torchvision.__version__);"
    "print('cuda        :', torch.cuda.is_available());"
    "print('GPU         :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A');"
    "print('unsloth     :', m.version('unsloth'));"
    "print('torchao     :', m.version('torchao'));"
    "print('transformers:', m.version('transformers'));"
    "print('trl         :', m.version('trl'))"],
    capture_output=True, text=True)
print("\n" + "=" * 56)
print(check.stdout or check.stderr)
print("=" * 56)
if check.returncode:
    print("Environment is broken — read the error above.")
    print("   If it mentions torch.int1 / _pytree.register_constant / torchvision")
    print("   extension, your torch is too old. Change the template; don't patch.")
else:
    print("Environment OK")
    print("If pip installed anything just now, RESTART THE KERNEL before Run All —")
    print("   packages changed on disk but this kernel still holds the old ones.")


## Imports

Imports the libraries and fixes random seeds so the sampling and training order are
reproducible.

In [ ]:
import os, io, re, json, random, shutil, time, subprocess, zipfile
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from tqdm.auto import tqdm
import torch

random.seed(SEED)
np.random.seed(SEED)
os.makedirs(WORK_DIR, exist_ok=True)
PNG_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"   torch {torch.__version__} | CUDA {torch.cuda.is_available()} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU'}")


## Fetch the image cache

Retrieves the cached shards produced by the cache builder and makes them available locally.

In [ ]:
# ============================================================
#  Fetch the prebuilt cache from Kaggle
# ============================================================
#  Built once by kaggle-chexpert-cache-builder.ipynb: every frontal CheXpert
#  image, already resized to exactly what LLaVA-1.5 consumes, plus the full
#  report table. Kaggle -> RunPod directly; your laptop is never involved.
#
#  Idempotent: if the cache is here it does nothing.
KAGGLE_TOKEN  = ""   # live credential — rotate when done
KAGGLE_KERNEL = "notebook571f9544b9"
UPLOAD_DIR    = Path(f"{WORK_DIR}/upload")
DELETE_ZIPS   = True   # extract-then-delete per shard: peak disk ~13 GB, not ~23 GB

def _dir_bytes(d):
    p = Path(d)
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) if p.exists() else 0

_have = len(list(PNG_CACHE_DIR.rglob("*.png")))
if _have > 150_000 and LOCAL_PARQUET.exists():
    print(f"Cache already present: {_have:,} images + parquet — nothing to fetch")
else:
    print(f"(have {_have:,} images, parquet={LOCAL_PARQUET.exists()}) — fetching")
    subprocess.run("pip install kaggle -q", shell=True)

    os.environ["KAGGLE_API_TOKEN"] = KAGGLE_TOKEN         # Kaggle's newer KGAT_ format
    _kd = Path.home() / ".kaggle"; _kd.mkdir(exist_ok=True)
    (_kd / "access_token").write_text(KAGGLE_TOKEN)
    (_kd / "access_token").chmod(0o600)

    UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
    EXPECTED = 11.6e9

    # `kaggle kernels output` prints nothing until each file completes — with 2 GB
    # shards that is minutes of silence, indistinguishable from a hang. Run it as a
    # child process and watch the directory grow instead.
    cmd = f"kaggle kernels output {KAGGLE_KERNEL} -p {UPLOAD_DIR}"
    print(f"\nPulling ~11.6 GB\n$ {cmd}", flush=True)
    t0   = time.time()
    proc = subprocess.Popen(cmd, shell=True)
    bar  = tqdm(total=int(EXPECTED), unit="B", unit_scale=True, unit_divisor=1000,
                desc="kaggle download", smoothing=0.1)
    try:
        while proc.poll() is None:
            got = _dir_bytes(UPLOAD_DIR)
            bar.n = min(got, int(EXPECTED))
            bar.set_postfix_str(f"{got/1e9:.2f} GB | {got/max(time.time()-t0,1)/1e6:.0f} MB/s")
            bar.refresh(); time.sleep(3)
    finally:
        bar.n = _dir_bytes(UPLOAD_DIR); bar.refresh(); bar.close()

    if proc.returncode:
        raise RuntimeError(f"kaggle kernels output failed (exit {proc.returncode}). "
                           "Check the token, and that the kernel version finished with output.")
    print(f"{_dir_bytes(UPLOAD_DIR)/1e9:.2f} GB in {(time.time()-t0)/60:.1f} min")

    _pq = list(UPLOAD_DIR.rglob("chexpert_plus_full.parquet"))
    if not _pq:
        raise FileNotFoundError(f"No parquet under {UPLOAD_DIR} — wrong kernel slug?")
    shutil.copy(_pq[0], LOCAL_PARQUET)
    print(f"{LOCAL_PARQUET} ({LOCAL_PARQUET.stat().st_size/1e6:.0f} MB)")

    _zips = sorted(UPLOAD_DIR.rglob("images_part*.zip"))
    print(f"\nExtracting {len(_zips)} shards -> {PNG_CACHE_DIR}")
    for z in tqdm(_zips, desc="extract", unit="shard"):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(PNG_CACHE_DIR)
        if DELETE_ZIPS:
            z.unlink()          # free 2 GB now, not at the end
    _have = len(list(PNG_CACHE_DIR.rglob("*.png")))

_gb = sum(p.stat().st_size for p in PNG_CACHE_DIR.rglob("*.png")) / 1e9
print(f"\nCache  : {_have:,} images | {_gb:.2f} GB | {PNG_CACHE_DIR}")
print(f" Reports: {LOCAL_PARQUET}")


## Key mapping and preprocessing

Normalises the report table's image paths into the key format used by the cache manifest, and
defines the image preprocessing. Path formats differ between the report table and the image
store, so this mapping is required for any image to be found at all.

In [ ]:
# ============================================================
#  Key mapping, preprocessing, and (lazy) Redivis
# ============================================================
import redivis

# ---------------------------------------------------------------------------
#  Redivis is a FALLBACK ONLY. With the cache present it is never contacted —
#  no query, no 223k-file listing, no rate limit, no token expiry. connect_redivis()
#  brings it up on demand if something genuinely turns out to be missing.
# ---------------------------------------------------------------------------
organization = None
dataset      = None

def connect_redivis():
    global organization, dataset
    if dataset is None:
        # to_directory() hands the raw urllib3 stream to pyarrow, and the API serves
        # the file listing with `x-accel-buffering: no`, so the body arrives in small
        # chunks. urllib3's read(n) returns only what has landed (~12 KB), pyarrow
        # wants a ~567 KB Arrow message, sees a short read, and calls the stream
        # truncated -> "OSError: Expected to be able to read 566984 bytes ... got 11853".
        # It's a race with the network, which is why it works some days and not others.
        import redivis.common.TabularReader as _TR
        if not getattr(_TR, "_shortread_patched", False):
            _orig = _TR.make_request
            def _buffered(*a, **k):
                r = _orig(*a, **k)
                p = k.get("path", "") or ""
                if isinstance(p, str) and p.endswith("/rawFiles") and getattr(r, "status_code", None) == 200:
                    r.raw = io.BytesIO(r.raw.read(decode_content=True))   # drain to EOF
                return r
            _TR.make_request = _buffered
            _TR._shortread_patched = True
        organization = redivis.organization("AIMI")
        dataset      = organization.dataset(DATASET_REF)
        dataset.get()
        print(f" Redivis: {dataset.properties.get('name')} "
              f"{dataset.properties.get('version', {}).get('tag')}")
    return dataset


def to_png_key(path_to_image: str) -> str:
    """
    Report table stores : train/patient00003/study1/view1_frontal.jpg
    PNG_train stores    : patient00003/study1/view1_frontal.png
    No split prefix, .png not .jpg. Getting this wrong silently matches nothing.
    """
    parts = list(Path(str(path_to_image).strip()).parts)
    while parts and (parts[0] in ("train", "valid", "test") or parts[0].startswith("CheXpert")):
        parts = parts[1:]
    return str(Path(*parts).with_suffix(".png"))


def clip_preprocess(img, size: int = None):
    """
    Byte-for-byte what llava-1.5-7b-hf's CLIPImageProcessor does
    (preprocessor_config.json: shortest_edge 336 -> bicubic -> center crop 336).
    Verified against the real processor: max|diff| == 0.000000 on every CheXpert size.

    NOTE int(), not round(). transformers computes the long edge as
    int(size * long / short) — truncation. round() gives 410 where transformers gives
    409 for a 2828x2320 image, shifting the crop a pixel. That is the most common
    CheXpert size, so the difference would touch most of the data.

    Grayscale: X-rays are single-channel, and resizing in 'L' then replicating to RGB
    is bit-identical to resizing in RGB (verified) at ~61 KB/image vs ~103 KB.
    """
    size = size or IMG_SIZE
    img  = img.convert("L")
    w, h = img.size
    short, lng = (w, h) if w <= h else (h, w)
    new_short, new_long = size, int(size * lng / short)
    new_w, new_h = (new_short, new_long) if w <= h else (new_long, new_short)
    img = img.resize((new_w, new_h), Image.BICUBIC)
    l, t = (new_w - size) // 2, (new_h - size) // 2
    return img.crop((l, t, l + size, t + size))


OFFLINE = len(list(PNG_CACHE_DIR.rglob("*.png"))) > 150_000 and LOCAL_PARQUET.exists()
print(f"{'OFFLINE — Redivis will not be contacted' if OFFLINE else 'cache incomplete — Redivis may be needed'}")
print(f"   key map: train/patient00003/study1/view1_frontal.jpg -> "
      f"{to_png_key('train/patient00003/study1/view1_frontal.jpg')}")


## Load the reports

Loads the report text that forms the training targets and removes empty or unusable entries.

In [ ]:
# ============================================================
#  Reports — from the local parquet (Redivis only if it's absent)
# ============================================================
RAW_REPORT_QUERY = f"""
SELECT
    path_to_image, path_to_dcm, deid_patient_id, patient_report_date_order,
    frontal_lateral, ap_pa, split, report,
    section_narrative, section_clinical_history, section_history,
    section_comparison, section_technique, section_procedure_comments,
    section_findings, section_impression, section_end_of_impression,
    section_summary, section_accession_number
FROM `{REPORT_TABLE}`
WHERE report IS NOT NULL
"""

if LOCAL_PARQUET.exists():
    print(f"Reading reports from {LOCAL_PARQUET} (offline)")
    raw_df = pd.read_parquet(LOCAL_PARQUET)
    raw_df = raw_df[raw_df["report"].notna()]
    SOURCE = str(LOCAL_PARQUET)
else:
    print(" No local parquet — querying Redivis")
    raw_df = connect_redivis().query(RAW_REPORT_QUERY).to_pandas_dataframe()
    SOURCE = REPORT_TABLE

print(f"{raw_df.shape[0]:,} rows x {raw_df.shape[1]} cols  (from {Path(SOURCE).name})")
print(f"   memory: {raw_df.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"\n Sample report:\n{'-'*60}")
print(str(raw_df['report'].dropna().iloc[0])[:400])


## Exclude the benchmark patients

Reads the evaluation benchmark and builds the set of patients that must never appear in
training. Without this the later evaluation would be measuring memorisation rather than
generalisation.

In [ ]:
# ============================================================
#  test_vqa.jsonl -> patient blacklist
# ============================================================
#  The one file Kaggle cannot supply: you generated it in your entity-extraction
#  notebook. Upload it to /workspace/data/test_vqa/ (15.7 MB).
#
#  This is load-bearing: these patients are removed BEFORE sampling, so training
#  never sees a test patient. Skipping it silently invalidates every number you
#  report later.
if Path(VQA_SRC).exists() and any(Path(VQA_SRC).iterdir()):
    shutil.copytree(VQA_SRC, VQA_DST, dirs_exist_ok=True)
    print(f"copied {VQA_SRC} -> {VQA_DST}")
else:
    found = list(Path(WORK_DIR).rglob("test_vqa.jsonl"))
    if not found:
        raise FileNotFoundError(
            f"test_vqa.jsonl not found anywhere under {WORK_DIR}.\n"
            f"Upload it via the Jupyter file browser into {VQA_SRC}/."
        )
    shutil.copytree(found[0].parent, VQA_DST, dirs_exist_ok=True)
    print(f"found and copied from {found[0].parent}")

vqa_jsonl_path = Path(VQA_DST) / "test_vqa.jsonl"
if not vqa_jsonl_path.exists():
    vqa_jsonl_path = list(Path(VQA_DST).rglob("test_vqa.jsonl"))[0]

vqa_records = [json.loads(l) for l in open(vqa_jsonl_path) if l.strip()]
blacklist_patients = {r["patient_id"] for r in vqa_records if "patient_id" in r}

print(f"{len(vqa_records):,} VQA rows | {len(blacklist_patients):,} blacklisted patients")
print(f"   sample: {sorted(blacklist_patients)[:5]}")


## Filter, clean and sample

Restricts the data to frontal training images with usable reports, removes the excluded
patients, and samples the target number of training pairs.

In [ ]:
# ============================================================
#  Filter, clean, sample
# ============================================================
print(f"Starting rows                : {len(raw_df):,}")
df = raw_df.copy()

df = df[df['frontal_lateral'].str.lower().str.strip() == 'frontal']
print(f"After frontal filter         : {len(df):,}")

df = df[df['split'].str.lower().str.strip() == 'train']
print(f"After train-split filter     : {len(df):,}")

_before = len(df)
df = df[~df['deid_patient_id'].isin(blacklist_patients)]
print(f"After blacklist removal      : {len(df):,}  (removed {_before-len(df):,})")

df = df[df['report'].notna()]
df = df[df['report'].str.strip().str.len() >= MIN_REPORT_LEN]
print(f"After report quality filter  : {len(df):,}")

df = df.drop_duplicates(subset=['deid_patient_id', 'patient_report_date_order'])
print(f"After dedup (patient+study)  : {len(df):,}")

# pandas' sample -> RandomState.choice(n, size, replace=False) -> permutation(n)[:size],
# so with a fixed seed the samples are NESTED: the 50k draw is exactly the first 50k
# of the 100k draw. Changing SAMPLE_SIZE therefore never needs new images.
if len(df) < SAMPLE_SIZE:
    print(f"only {len(df):,} available — using all")
    df_sampled = df.reset_index(drop=True)
else:
    df_sampled = df.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)

print(f"\nFinal dataset : {len(df_sampled):,} rows")
print(f"   unique patients: {df_sampled['deid_patient_id'].nunique():,}")
print(f"   report chars   : mean {df_sampled['report'].str.len().mean():.0f} | "
      f"p95 {df_sampled['report'].str.len().quantile(.95):.0f} | "
      f"max {df_sampled['report'].str.len().max():.0f}")


## Image fetcher

Defines the routine that reads an image by key from the cached shards and applies preprocessing,
used by both the dataset and the later evaluation so both see identical inputs.

In [ ]:
# ============================================================
#  The image fetcher — used by viz, training and eval alike
# ============================================================
_png_table = None

def _table():
    global _png_table
    if _png_table is None:
        _png_table = connect_redivis().table(PNG_TRAIN_REF)
        _png_table.get()
    return _png_table

def fetch_pil_image(image_path: str, mode: str = "RGB", retries: int = 3):
    """
    336x336 image for a CheXpert path. Local cache hit (~2.5 ms) in the normal case;
    the network path is a fallback for a miss and costs ~700 ms.
    Returns None only if the image genuinely does not exist.
    """
    key    = to_png_key(image_path)
    cached = PNG_CACHE_DIR / key
    if cached.exists():
        try:
            return Image.open(cached).convert(mode)
        except Exception:
            cached.unlink(missing_ok=True)      # corrupt entry -> refetch

    for attempt in range(retries):
        try:
            raw = _table().file(key).read()
            img = clip_preprocess(Image.open(io.BytesIO(raw)))
            cached.parent.mkdir(parents=True, exist_ok=True)
            tmp = cached.with_suffix(".tmp")
            img.save(tmp, format="PNG"); tmp.replace(cached)   # atomic
            return img.convert(mode)
        except Exception as e:
            if type(e).__name__ == "NotFoundError":
                return None
            if attempt == retries - 1:
                raise
            time.sleep(1.5 * (attempt + 1))
    return None

_p = df_sampled['path_to_image'].iloc[0]
_i = fetch_pil_image(_p)
print(f"Smoke test: {_p}\n  -> {to_png_key(_p)} -> "
      f"{f'{_i.size} {_i.mode} std={np.asarray(_i).std():.1f} ' if _i else 'FAILED'}")


## Visual check

Displays a few sampled images next to their reports. This is the check that no assertion can
perform: it confirms the pairing is correct and the images are readable before hours of training
are committed.

In [ ]:
# ============================================================
#  Visual sanity check: images <-> reports
# ============================================================
#  This is the cheapest guard you have against a silent pairing bug. Look at it:
#  a report mentioning a pacemaker should sit next to an X-ray with a pacemaker.
VIZ_N   = 8
VIZ_OUT = Path(WORK_DIR) / "llava_s_sample_viz.png"

rows = []
for _, row in df_sampled.sample(frac=1, random_state=SEED).iterrows():
    img = fetch_pil_image(row['path_to_image'], mode="L")
    if img is not None:
        rows.append((row, img))
    if len(rows) == VIZ_N:
        break
print(f"loaded {len(rows)}/{VIZ_N} images")
if not rows:
    raise RuntimeError("Zero images loaded — check to_png_key() against the cache layout")

fig   = plt.figure(figsize=(22, len(rows) * 5.5), facecolor='#0d0d0d')
outer = gridspec.GridSpec(len(rows), 2, figure=fig, wspace=0.03, hspace=0.38,
                          width_ratios=[1, 2.4])
for i, (row, img) in enumerate(rows):
    ax = fig.add_subplot(outer[i, 0])
    ax.imshow(img, cmap='bone', aspect='auto')
    ax.set_title(f"#{i+1} | {row['deid_patient_id']} | {row.get('ap_pa','?')}",
                 fontsize=7.5, color='#90caf9', pad=3)
    for sp in ax.spines.values():
        sp.set_edgecolor('#1565C0'); sp.set_linewidth(1.4)
    ax.set_xticks([]); ax.set_yticks([])

    axt = fig.add_subplot(outer[i, 1]); axt.set_facecolor('#111827'); axt.axis('off')
    for sp in axt.spines.values():
        sp.set_edgecolor('#374151'); sp.set_linewidth(0.8)
    rep = str(row['report']).strip()
    axt.text(0.01, 0.98, to_png_key(row['path_to_image']), transform=axt.transAxes,
             fontsize=6.2, color='#6ee7b7', fontweight='bold', va='top', fontfamily='monospace')
    axt.text(0.01, 0.90, rep[:750] + ('…' if len(rep) > 750 else ''), transform=axt.transAxes,
             fontsize=6.8, color='#e5e7eb', va='top', fontfamily='monospace')
    axt.text(0.01, 0.03, f"report_len={len(rep)}c | study={row.get('patient_report_date_order','?')}",
             transform=axt.transAxes, fontsize=5.8, color='#9ca3af', va='bottom', fontfamily='monospace')

plt.suptitle(f'LLaVA-S — Frontal X-rays + Raw Reports ({len(rows)} samples)',
             fontsize=12, fontweight='bold', color='white', y=1.002)
fig.patch.set_facecolor('#0d0d0d')
plt.tight_layout()
plt.savefig(VIZ_OUT, dpi=130, bbox_inches='tight', facecolor='#0d0d0d')
plt.close(fig)
print(f"{VIZ_OUT} ({VIZ_OUT.stat().st_size/1e6:.2f} MB)")
try:
    from IPython.display import display, Image as IPyImage
    display(IPyImage(filename=str(VIZ_OUT), width=1100))
except Exception:
    pass


## Build the instruction dataset

Converts the image-report pairs into the instruction format the trainer expects: an image with a
prompt asking for a description, and the report as the target response.

In [ ]:
# ============================================================
#  Build the instruction dataset
# ============================================================
INSTRUCTION = "Describe the findings in this chest X-ray."

def make_record(row) -> dict:
    path    = str(row['path_to_image']).strip()
    parts   = Path(path).parts
    patient = next((p for p in parts if p.startswith('patient')), str(row['deid_patient_id']))
    study   = next((p for p in parts if p.startswith('study')), 'study_unknown')
    report  = str(row['report']).strip()
    return {
        "id"           : f"{patient}_{study}_{Path(path).stem}",
        "patient_id"   : str(row['deid_patient_id']).strip(),
        "image_path"   : path,
        "report_len"   : len(report),
        "conversations": [
            {"from": "human", "value": f"<image>\n{INSTRUCTION}"},
            {"from": "gpt",   "value": report},
        ],
    }

records = [make_record(r) for _, r in tqdm(df_sampled.iterrows(), total=len(df_sampled),
                                           desc="records")]
print(f"{len(records):,} records")

with open(DATASET_JSONL, 'w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print(f"{DATASET_JSONL} ({Path(DATASET_JSONL).stat().st_size/1e6:.1f} MB)")

pd.DataFrame([{k: r[k] for k in ('id','patient_id','image_path','report_len')}
              for r in records]).to_parquet(DATASET_PARQUET, index=False)
print(f"{DATASET_PARQUET}")

_s = dict(records[0]); _s['conversations'] = [_s['conversations'][0],
    {"from":"gpt","value": _s['conversations'][1]['value'][:200] + '...'}]
print("\n Sample record:"); print(json.dumps(_s, indent=2))


## Integrity checks

Confirms every sample resolves to a real image, the fields are well-formed, and no excluded
patient is present.

In [ ]:
# ============================================================
#  Integrity checks — all must pass before training
# ============================================================
ok = True

overlap = {r['patient_id'] for r in records} & blacklist_patients
if overlap:
    print(f"CHECK 1 FAILED — {len(overlap)} patients in BOTH train and test: {sorted(overlap)[:5]}")
    ok = False
else:
    print(f"CHECK 1 — no train/test patient overlap "
          f"({len({r['patient_id'] for r in records}):,} train vs {len(blacklist_patients):,} test)")

non_frontal = [r for r in records if 'frontal' not in r['image_path'].lower()]
print(f"{'' if not non_frontal else ''} CHECK 2 — {len(records)-len(non_frontal):,}/{len(records):,} paths contain 'frontal'")

short = [r for r in records if r['report_len'] < MIN_REPORT_LEN]
if short:
    print(f"CHECK 3 FAILED — {len(short)} reports under {MIN_REPORT_LEN} chars"); ok = False
else:
    print(f"CHECK 3 — every report >= {MIN_REPORT_LEN} chars "
          f"(min {min(r['report_len'] for r in records)})")

missing = [r for r in records if not (PNG_CACHE_DIR / to_png_key(r['image_path'])).exists()]
print(f"{'' if not missing else ''} CHECK 4 — {len(records)-len(missing):,}/{len(records):,} images cached"
      f"{'' if not missing else f' ({len(missing):,} missing — the prefetch cell will fetch them)'}")

print("\n" + ("ALL CHECKS PASSED" if ok else "FIX THE FAILURES ABOVE BEFORE TRAINING"))


## Prefetch

Ensures every image required for training is present locally, so the training loop is never
blocked on a fetch. This is a no-op if the cache is already complete.

In [ ]:
# ============================================================
#  Prefetch — only runs if the Kaggle cache is incomplete
# ============================================================
#  Normally a no-op: the Kaggle bundle already holds every frontal image. This is
#  the safety net for a partial cache. If it does run, it is network-bound and
#  Redivis caps you at ~16.7 files/s, so more threads past ~24 just earn 429s.
from concurrent.futures import ThreadPoolExecutor
import threading

_needed = [(r["id"], to_png_key(r["image_path"])) for r in records]
_todo   = [(i, k) for i, k in _needed if not (PNG_CACHE_DIR / k).exists()]

if not _todo:
    print(f"All {len(_needed):,} images cached — Redivis not contacted")
else:
    print(f"{len(_todo):,} images missing — fetching from Redivis "
          f"(~{len(_todo)*3.29/1000:.1f} GB over the wire)")
    png = connect_redivis().table(PNG_TRAIN_REF); png.get()
    print("indexing PNG_train (one time, ~40s)...")
    by_key = {str(f.path): f for f in png.list_files()}

    lock, prog, failures = threading.Lock(), {"ok": 0, "fail": 0}, []
    def _one(item):
        _id, key = item
        dest = PNG_CACHE_DIR / key
        if dest.exists() or key not in by_key:
            return
        for a in range(4):
            try:
                img = clip_preprocess(Image.open(io.BytesIO(by_key[key].read())))
                dest.parent.mkdir(parents=True, exist_ok=True)
                tmp = dest.with_suffix(".tmp"); img.save(tmp, "PNG"); tmp.replace(dest)
                with lock: prog["ok"] += 1
                return
            except Exception as e:
                if a == 3:
                    with lock: prog["fail"] += 1; failures.append((key, repr(e)[:80]))
                else: time.sleep(1.5 * (a + 1))

    with ThreadPoolExecutor(max_workers=PREFETCH_WORKERS) as ex:
        list(tqdm(ex.map(_one, _todo), total=len(_todo), desc="prefetch", unit="img"))
    print(f"fetched {prog['ok']:,} | failed {prog['fail']:,}")
    for k, e in failures[:5]: print(f"   {k}: {e}")

    _gone = {k for k, _ in failures} | {k for _, k in _needed if k not in by_key}
    if _gone:
        _b = len(records)
        records = [r for r in records if to_png_key(r["image_path"]) not in _gone]
        print(f"dropped {_b-len(records):,} records with no fetchable image")
        with open(DATASET_JSONL, "w", encoding="utf-8") as f:
            for r in records: f.write(json.dumps(r, ensure_ascii=False) + "\n")

_have = sum(1 for r in records if (PNG_CACHE_DIR / to_png_key(r["image_path"])).exists())
print(f"\n{_have:,}/{len(records):,} training images on local disk")
if _have != len(records):
    raise RuntimeError(f"{len(records)-_have} records still have no image — re-run this cell")
print("Every training record has a real cached X-ray")


## Load the model

Loads LLaVA-1.5-7B quantised to 4-bit and attaches LoRA adapters. Only the adapters are trained;
the base weights stay frozen, which is what makes finetuning a 7B model feasible on a single
GPU.

In [ ]:
# ============================================================
#  Load LLaVA-1.5-7B via Unsloth (4-bit QLoRA)
# ============================================================
from unsloth import FastVisionModel
import torch

print(f"Loading {BASE_MODEL} (4-bit={LOAD_IN_4BIT})")
print("   first run downloads ~14 GB (or ~4 GB for the unsloth bnb-4bit repo)")

model, tokenizer = FastVisionModel.from_pretrained(
    model_name   = BASE_MODEL,
    load_in_4bit = LOAD_IN_4BIT,
    use_gradient_checkpointing = "unsloth",
)
print("base model loaded")

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r            = LORA_R,
    lora_alpha   = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias         = "none",
    use_rslora   = False,
    random_state = SEED,
)
print("LoRA applied")
model.print_trainable_parameters()
print(f"   VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated")


## Dataset, collator and preflight

Wires the dataset to the trainer's collator and runs a single batch through the model before
training starts, so shape or token-count errors surface immediately.

In [ ]:
# ============================================================
#  Torch dataset + collator + preflight
# ============================================================
from torch.utils.data import Dataset as TorchDataset
from unsloth.trainer import UnslothVisionDataCollator
from transformers import AutoProcessor

raw_records = [json.loads(l) for l in open(DATASET_JSONL) if l.strip()]
print(f"{len(raw_records):,} records from {DATASET_JSONL}")

class CheXpertLLaVADataset(TorchDataset):
    """Images come from the local 336x336 cache (~2.5 ms), reports from the JSONL."""
    def __init__(self, records, tokenizer, processor, max_seq_len=MAX_SEQ_LEN):
        self.records, self.tokenizer = records, tokenizer
        self.processor, self.max_seq_len = processor, max_seq_len

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        img = fetch_pil_image(rec['image_path'])

        # NEVER substitute a blank image here. The original notebook fell back to a
        # black 224x224 on failure — combined with a broken key mapping that meant
        # every sample was blank, and training would have completed with a healthy
        # loss curve and produced a model that never saw an X-ray. Fail loudly.
        if img is None:
            raise RuntimeError(
                f"No image for {rec['id']!r}\n  path: {rec['image_path']}\n"
                f"  key : {to_png_key(rec['image_path'])}"
            )
        return {"messages": [
            {"role": "user", "content": [{"type": "image", "image": img},
                                         {"type": "text",  "text": INSTRUCTION}]},
            {"role": "assistant", "content": [{"type": "text",
                                               "text": rec['conversations'][1]['value']}]},
        ]}

processor     = AutoProcessor.from_pretrained(BASE_MODEL)
train_dataset = CheXpertLLaVADataset(raw_records, tokenizer, processor)
data_collator = UnslothVisionDataCollator(model, tokenizer)
print(f"dataset ({len(train_dataset):,}) + collator ready")

# ---- Preflight: prove real pixels reach the model before burning GPU hours ----
# Catches a broken key mapping in seconds instead of after a full epoch.
print("\npreflight — 5 random samples")
for i in random.Random(SEED).sample(range(len(train_dataset)), k=min(5, len(train_dataset))):
    im  = train_dataset[i]['messages'][0]['content'][0]['image']
    arr = np.asarray(im)
    if arr.std() < 1.0:
        raise RuntimeError(f"sample {i} is flat/blank (std={arr.std():.3f}) — image fetch is broken")
    print(f"   [{i:6d}] {im.size} {im.mode} std={arr.std():5.1f} ")

# The processor's crop should be a no-op on our cache — confirm the sizes agree.
_cs = processor.image_processor.crop_size
assert (_cs['height'], _cs['width']) == (IMG_SIZE, IMG_SIZE), \
    f"processor expects {_cs}, cache is {IMG_SIZE}x{IMG_SIZE}"
print(f"preflight passed — real X-rays, and processor crop_size {_cs} matches the cache")


## Train

Runs supervised finetuning for one epoch, saving checkpoints periodically so an interrupted run
can be resumed.

In [ ]:
# ============================================================
#  Train
# ============================================================
import dataclasses, inspect, math
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

torch.backends.cuda.matmul.allow_tf32 = True     # free on Ampere
torch.backends.cudnn.allow_tf32       = True
torch.backends.cudnn.benchmark        = True

_bf16 = torch.cuda.is_bf16_supported()           # True on the 3090
_steps = math.ceil(len(train_dataset) / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
print(f"schedule: {len(train_dataset):,} samples | {BATCH_SIZE}x{GRAD_ACCUM} "
      f"= {BATCH_SIZE*GRAD_ACCUM} effective | ~{_steps:,} steps | {'bf16' if _bf16 else 'fp16'}")

_cfg = dict(
    output_dir                  = ADAPTER_SAVE_DIR,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    num_train_epochs            = EPOCHS,
    learning_rate               = LR,
    lr_scheduler_type           = LR_SCHEDULER,
    warmup_ratio                = WARMUP_RATIO,
    fp16                        = not _bf16,
    bf16                        = _bf16,
    optim                       = "adamw_8bit",
    logging_steps               = 20,
    save_steps                  = 500,
    save_total_limit            = 2,
    seed                        = SEED,
    dataloader_num_workers      = DATALOADER_WORKERS,
    dataloader_pin_memory       = True,
    report_to                   = "none",
    remove_unused_columns       = False,
    dataset_kwargs              = {"skip_prepare_dataset": True},
)
if DATALOADER_WORKERS > 0:
    _cfg["dataloader_persistent_workers"] = True
    _cfg["dataloader_prefetch_factor"]    = 4

# TRL renamed SFTConfig.max_seq_length -> max_length (~0.20) and
# SFTTrainer(tokenizer=) -> processing_class. Bind to whatever this pod resolved
# rather than guessing and dying 40 minutes into a run.
_fields = {f.name for f in dataclasses.fields(SFTConfig)}
if   "max_seq_length" in _fields: _cfg["max_seq_length"] = MAX_SEQ_LEN
elif "max_length"     in _fields: _cfg["max_length"]     = MAX_SEQ_LEN
_dropped = sorted(set(_cfg) - _fields)
if _dropped: print(f"this TRL ignores: {_dropped}")
training_args = SFTConfig(**{k: v for k, v in _cfg.items() if k in _fields})

_tkw = dict(model=model, data_collator=data_collator,
            train_dataset=train_dataset, args=training_args)
if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters:
    _tkw["processing_class"] = tokenizer
else:
    _tkw["tokenizer"] = tokenizer
trainer = SFTTrainer(**_tkw)

# ---- Resume. ~25k steps is ~24h unattended; losing the pod is a real risk that
# has nothing to do with the code. save_steps=500 only helps if something reads
# the checkpoints back — trainer.train() with no argument silently restarts at 0.
_ck = sorted((p for p in Path(ADAPTER_SAVE_DIR).glob("checkpoint-*") if p.is_dir()),
             key=lambda p: int(p.name.split("-")[1])) if Path(ADAPTER_SAVE_DIR).exists() else []
_resume = str(_ck[-1]) if _ck else None
print(f"{'RESUMING from ' + _ck[-1].name if _resume else 'starting fresh'}")

print("=" * 56)
print("The tqdm ETA after ~50 steps is the number to trust, not any estimate.")
print("=" * 56)
train_result = trainer.train(resume_from_checkpoint=_resume)

print("=" * 56)
print(f"done | loss {train_result.training_loss:.4f} | steps {train_result.global_step:,} "
      f"| {train_result.metrics.get('train_runtime',0)/3600:.2f} h "
      f"| peak VRAM {torch.cuda.max_memory_allocated()/1e9:.1f} GB")


## Save the adapter

Writes the trained LoRA adapter and tokenizer files. This adapter is the LLaVA-S model and is
the starting point for the V-RAG finetuning stage.

In [ ]:
# ============================================================
#  Save the LoRA adapter
# ============================================================
os.makedirs(ADAPTER_SAVE_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_SAVE_DIR)
tokenizer.save_pretrained(ADAPTER_SAVE_DIR)

_files = [f for f in Path(ADAPTER_SAVE_DIR).iterdir() if f.is_file()]
print(f"adapter -> {ADAPTER_SAVE_DIR}  ({sum(f.stat().st_size for f in _files)/1e6:.1f} MB)")
for f in sorted(_files):
    print(f"   {f.name:44s} {f.stat().st_size/1e6:7.2f} MB")
print("\nThis adapter is the LLaVA_S backbone for the three V-RAG finetunes.")
print("It is small and portable — download it and evaluate anywhere.")


## Sanity inference

Generates a report for a held-out image to confirm the trained model loads and produces
radiology-style output rather than generic image description.

In [ ]:
# ============================================================
#  Sanity inference on held-out images
# ============================================================
FastVisionModel.for_inference(model)

_train_paths = {r['image_path'] for r in raw_records}
held = raw_df[
    (raw_df['frontal_lateral'].str.lower().str.strip() == 'frontal') &
    (raw_df['split'].str.lower().str.strip() == 'train') &
    (~raw_df['deid_patient_id'].isin(blacklist_patients)) &
    (~raw_df['path_to_image'].isin(_train_paths)) &
    (raw_df['report'].notna()) &
    (raw_df['report'].str.len() > MIN_REPORT_LEN)
].sample(n=3, random_state=SEED + 1).reset_index(drop=True)
print(f"{len(held)} held-out images (not trained on, not in test blacklist)")

fig, axes = plt.subplots(3, 2, figsize=(18, 18), facecolor='#0d0d0d')
for i, row in held.iterrows():
    img = fetch_pil_image(row['path_to_image'])
    if img is None:
        # A blank here would produce a confident report about an X-ray the model
        # never saw — the same trap as the training wrapper.
        raise RuntimeError(f"no image for held-out {row['path_to_image']}")

    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": INSTRUCTION}]}],
        add_generation_prompt=True)
    inputs = tokenizer(images=img, text=text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True)
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    ax = axes[i, 0]; ax.imshow(img, cmap='bone'); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{row['deid_patient_id']}", fontsize=8, color='#90caf9')
    axt = axes[i, 1]; axt.axis('off'); axt.set_facecolor('#111827')
    axt.text(0.01, 0.99, "GENERATED", transform=axt.transAxes, fontsize=7,
             color='#6ee7b7', fontweight='bold', va='top', fontfamily='monospace')
    axt.text(0.01, 0.94, gen[:700], transform=axt.transAxes, fontsize=6.5,
             color='#e5e7eb', va='top', fontfamily='monospace')
    axt.text(0.01, 0.46, "GROUND TRUTH", transform=axt.transAxes, fontsize=7,
             color='#fbbf24', fontweight='bold', va='top', fontfamily='monospace')
    axt.text(0.01, 0.41, str(row['report']).strip()[:700], transform=axt.transAxes,
             fontsize=6.5, color='#9ca3af', va='top', fontfamily='monospace')
    print(f"\n--- {row['deid_patient_id']} ---\nGEN: {gen[:220]}\nGT : {str(row['report'])[:220]}")

fig.patch.set_facecolor('#0d0d0d'); plt.tight_layout()
_out = Path(WORK_DIR) / "llava_s_sanity_inference.png"
plt.savefig(_out, dpi=120, bbox_inches='tight', facecolor='#0d0d0d'); plt.close(fig)
print(f"\n{_out}")
try:
    from IPython.display import display, Image as IPyImage
    display(IPyImage(filename=str(_out), width=1100))
except Exception:
    pass


In [ ]:
# ============================================================
#   CELL 13C — Visualize Test Set Samples
#   Shows each image + all its entity questions
# ============================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

print("=" * 62)
print("  CELL 13C — Visualizing Test Set Samples")
print("=" * 62)

# Pick EVAL_VIZ_N images that have the most entity questions
top_image_ids = sorted(
    image_to_vqas.keys(),
    key=lambda iid: len(image_to_vqas[iid]),
    reverse=True
)[:EVAL_VIZ_N]

fig = plt.figure(figsize=(22, EVAL_VIZ_N * 5.5), facecolor='#0d0d0d')
outer = gridspec.GridSpec(EVAL_VIZ_N, 2, figure=fig,
                          wspace=0.03, hspace=0.4,
                          width_ratios=[1, 2.4])

for i, image_id in enumerate(top_image_ids):
    vqas = image_to_vqas[image_id]
    path = vqas[0]["path"]

    img, dbg = load_test_image(path)

    # --- Image panel ---
    ax_img = fig.add_subplot(outer[i, 0])
    if img is not None:
        ax_img.imshow(img, cmap='bone', aspect='auto')
        border = '#1565C0'
    else:
        ax_img.set_facecolor('#1a1a1a')
        ax_img.text(0.5, 0.5, f"[ Not loaded ]\n{dbg[:60]}",
                    ha='center', va='center', color='#ef4444',
                    fontsize=7, transform=ax_img.transAxes,
                    fontfamily='monospace')
        border = '#7f1d1d'

    ax_img.set_title(
        f"#{i+1} | {vqas[0]['patient_id']}\n{path}",
        fontsize=6.5, color='#90caf9', pad=3
    )
    for sp in ax_img.spines.values():
        sp.set_edgecolor(border); sp.set_linewidth(1.4)
    ax_img.set_xticks([]); ax_img.set_yticks([])

    # --- Questions panel ---
    ax_txt = fig.add_subplot(outer[i, 1])
    ax_txt.set_facecolor('#111827')
    for sp in ax_txt.spines.values():
        sp.set_edgecolor('#374151'); sp.set_linewidth(0.8)
    ax_txt.axis('off')

    ax_txt.text(0.01, 0.99,
                f"ENTITY PROBING QUESTIONS  |  {len(vqas)} entities for this image",
                transform=ax_txt.transAxes, fontsize=7, color='#6ee7b7',
                fontweight='bold', va='top', fontfamily='monospace')

    lines = []
    for j, vqa in enumerate(vqas[:18]):
        tag = "YES " if vqa['answer'].lower() == 'yes' else "NO  "
        lines.append(f"  Q{j+1:02d}: Does the patient have [{vqa['entity']}]?")
        lines.append(f"        GT: {tag}")
    if len(vqas) > 18:
        lines.append(f"  ... +{len(vqas)-18} more entities")

    ax_txt.text(0.01, 0.91, "\n".join(lines),
                transform=ax_txt.transAxes, fontsize=6.2, color='#e5e7eb',
                va='top', fontfamily='monospace', multialignment='left')

    caption_preview = str(vqas[0].get('caption', '')).strip()[:200].replace('\n', ' ')
    ax_txt.text(0.01, 0.04,
                f"Caption: {caption_preview}…",
                transform=ax_txt.transAxes, fontsize=5.5, color='#6b7280',
                va='bottom', fontfamily='monospace')

plt.suptitle(
    f'Entity Probing Test Set — {EVAL_VIZ_N} Sample Images + Disease Entity Questions',
    fontsize=12, fontweight='bold', color='white', y=1.002
)
fig.patch.set_facecolor('#0d0d0d')
plt.tight_layout()
plt.savefig(EVAL_VIZ_OUT, dpi=130, bbox_inches='tight',
            facecolor='#0d0d0d', edgecolor='none')
plt.close(fig)
print(f"Saved -> {EVAL_VIZ_OUT}")

try:
    from IPython.display import display as ipy_display, Image as IPyImage
    ipy_display(IPyImage(filename=str(EVAL_VIZ_OUT), width=1100))
except Exception:
    pass

## Reset previous results

Clears any results file from an earlier evaluation so a rerun cannot append to stale output.

In [ ]:
# Safe reset of eval results before a fresh evaluation run (won't error if the file
# doesn't exist yet — the original hardcoded !rm would crash on a first run).
from pathlib import Path as _Path
_p = _Path(EVAL_RESULTS_JSONL)
if _p.exists():
    _p.unlink()
    print(f"  Removed old {EVAL_RESULTS_JSONL}")
else:
    print(f"No existing {EVAL_RESULTS_JSONL} — starting fresh")


## Run entity-probing inference

Asks the model each Yes/No question and records the parsed answer alongside the ground truth.

In [ ]:
# ============================================================
#   CELL 13D — Run Entity Probing Inference on LLaVA-S
# ============================================================

import re, torch, json
from pathlib import Path
from tqdm.notebook import tqdm as tqdm_nb
from transformers import AutoProcessor

print("=" * 62)
print("  CELL 13D — Entity Probing Inference")
print("=" * 62)

# == Switch to inference mode ==================================
FastVisionModel.for_inference(model)

# == Load processor (handles both image + text tokenization) ==
processor = AutoProcessor.from_pretrained(BASE_MODEL)
print(f"Processor loaded from {BASE_MODEL}")

# == Prompt ====================================================
def make_probe_prompt(entity: str) -> str:
    return (
        f"Answer the question with only the word yes or no. "
        f"Do not provide explanations. "
        f"According to the image, does the patient have {entity}?"
    )

def parse_yes_no(text: str) -> str:
    t = text.strip().lower()
    if t in ('yes', 'no'):
        return t.capitalize()
    first = t.split()[0] if t.split() else ''
    if first in ('yes', 'no'):
        return first.capitalize()
    if re.search(r'\byes\b', t):
        return 'Yes'
    if re.search(r'\bno\b', t):
        return 'No'
    return 'Unknown'

# == Single-image inference helper ============================
def run_single_probe(img, prompt: str) -> tuple:
    """
    Returns (pred_answer, raw_output).
    Uses processor — the correct way for LLaVA-1.5.
    """
    # LLaVA-1.5 conversation format
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    # apply_chat_template gives the text prompt with <image> token
    text_prompt = processor.apply_chat_template(
        conversation, add_generation_prompt=True
    )

    # processor handles both image preprocessing + tokenization
    inputs = processor(
        images  = img,
        text    = text_prompt,
        return_tensors = "pt",
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens = 10,    # only need yes / no
            do_sample      = False, # greedy
        )

    # decode only the newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    raw_out    = processor.decode(new_tokens, skip_special_tokens=True).strip()
    pred       = parse_yes_no(raw_out)
    return pred, raw_out

# == Quick inference smoke test before full loop ===============
print("\n Smoke test — probing first image in test set...")
_smoke_vqa  = vqa_records[0]
_smoke_img, _smoke_dbg = load_test_image(_smoke_vqa["path"])

if _smoke_img is None:
    raise RuntimeError(
        f"Image still failing to load.\n"
        f"Path : {_smoke_vqa['path']}\n"
        f"Debug: {_smoke_dbg}\n"
        f"Fix load_test_image() before running the full loop."
    )

_smoke_pred, _smoke_raw = run_single_probe(
    _smoke_img, make_probe_prompt(_smoke_vqa["entity"])
)
print(f"  Image path   : {_smoke_vqa['path']}")
print(f"  Entity       : {_smoke_vqa['entity']}")
print(f"  GT answer    : {_smoke_vqa['answer']}")
print(f"  Model raw    : '{_smoke_raw}'")
print(f"  Parsed pred  : {_smoke_pred}")
print(f"  Image debug  : {_smoke_dbg}")
print("Smoke test passed — starting full loop\n")

# == Resume support ============================================
done_ids = set()
if Path(EVAL_RESULTS_JSONL).exists():
    with open(EVAL_RESULTS_JSONL, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                done_ids.add(json.loads(line)['vqa_id'])
    print(f"⏩ Resuming — {len(done_ids):,} already done")

image_ids_to_run = [
    iid for iid in image_to_vqas.keys()
    if any(v['vqa_id'] not in done_ids for v in image_to_vqas[iid])
]
total_vqa_to_run = sum(
    len([v for v in image_to_vqas[i] if v['vqa_id'] not in done_ids])
    for i in image_ids_to_run
)

print(f"   Images to probe  : {len(image_ids_to_run):,}")
print(f"   VQA pairs to run : {total_vqa_to_run:,}")
print(f"   Already done     : {len(done_ids):,}\n")

# == Main inference loop =======================================
results_file  = open(EVAL_RESULTS_JSONL, 'a')
failed_images = []
n_written     = 0

for image_id in tqdm_nb(image_ids_to_run, desc="Probing images"):
    vqas     = image_to_vqas[image_id]
    img_path = vqas[0]["path"]

    # Load image ONCE for all entities of this image
    img, load_dbg = load_test_image(img_path)

    if img is None:
        failed_images.append((image_id, img_path, load_dbg))
        for vqa in vqas:
            if vqa['vqa_id'] in done_ids:
                continue
            rec = {
                "vqa_id"      : vqa['vqa_id'],
                "image_id"    : image_id,
                "patient_id"  : vqa['patient_id'],
                "entity"      : vqa['entity'],
                "gt_answer"   : vqa['answer'],
                "pred_raw"    : "IMAGE_LOAD_FAILED",
                "pred_answer" : "Unknown",
                "img_loaded"  : False,
            }
            results_file.write(json.dumps(rec) + '\n')
            n_written += 1
        results_file.flush()
        continue

    # Ask every entity question for this image
    for vqa in vqas:
        if vqa['vqa_id'] in done_ids:
            continue

        prompt = make_probe_prompt(vqa['entity'])

        try:
            pred, raw_out = run_single_probe(img, prompt)
        except Exception as e:
            raw_out = f"ERROR: {str(e)[:120]}"
            pred    = "Unknown"

        rec = {
            "vqa_id"      : vqa['vqa_id'],
            "image_id"    : image_id,
            "patient_id"  : vqa['patient_id'],
            "entity"      : vqa['entity'],
            "gt_answer"   : vqa['answer'],
            "pred_raw"    : raw_out,
            "pred_answer" : pred,
            "img_loaded"  : True,
        }
        results_file.write(json.dumps(rec) + '\n')
        n_written += 1

    results_file.flush()

results_file.close()

print(f"\nInference complete")
print(f"   Written this run   : {n_written:,}")
print(f"   Failed image loads : {len(failed_images):,}")
if failed_images:
    print(f"\n   Failed paths (first 5):")
    for iid, path, dbg in failed_images[:5]:
        print(f"     {iid}")
        print(f"     {dbg[:120]}")
print(f"\nSaved -> {EVAL_RESULTS_JSONL}")

## Compute the metrics

Computes precision, recall and F1 against the benchmark answers. In this task a false positive
is the model asserting a finding that is not present, which is the quantity of interest when
measuring hallucination.

In [ ]:
# ============================================================
#   CELL 13E — Compute Precision, Recall, F1
# ============================================================

from collections import defaultdict
import pandas as pd

print("=" * 62)
print("  CELL 13E — Computing Evaluation Metrics")
print("=" * 62)

# Reload all results from file (handles resume)
all_results = []
with open(EVAL_RESULTS_JSONL, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            all_results.append(json.loads(line))

print(f"Total results loaded : {len(all_results):,}")

# Filter out Unknown
valid   = [r for r in all_results if r['pred_answer'] in ('Yes', 'No')]
skipped = len(all_results) - len(valid)
if skipped:
    print(f"Skipping {skipped} Unknown predictions from metrics")

# == Overall confusion matrix ==================================
TP = sum(1 for r in valid if r['gt_answer'] == 'Yes' and r['pred_answer'] == 'Yes')
FP = sum(1 for r in valid if r['gt_answer'] == 'No'  and r['pred_answer'] == 'Yes')
FN = sum(1 for r in valid if r['gt_answer'] == 'Yes' and r['pred_answer'] == 'No')
TN = sum(1 for r in valid if r['gt_answer'] == 'No'  and r['pred_answer'] == 'No')

precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
accuracy  = (TP + TN) / len(valid) if valid else 0.0

print(f"\n{'='*52}")
print(f"  OVERALL RESULTS — LLaVA-S Entity Probing")
print(f"{'='*52}")
print(f"  Total evaluated  : {len(valid):,}")
print(f"  TP  : {TP:,}    FP  : {FP:,}")
print(f"  FN  : {FN:,}    TN  : {TN:,}")
print(f"{'='*52}")
print(f"  Precision : {precision:.4f}  ({precision*100:.2f}%)")
print(f"  Recall    : {recall:.4f}  ({recall*100:.2f}%)")
print(f"  F1 Score  : {f1:.4f}  ({f1*100:.2f}%)")
print(f"  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"{'='*52}")

# == Per-entity breakdown ======================================
entity_stats = defaultdict(lambda: {'TP':0,'FP':0,'FN':0,'TN':0,'total':0})
for r in valid:
    e = r['entity']
    entity_stats[e]['total'] += 1
    if   r['gt_answer']=='Yes' and r['pred_answer']=='Yes': entity_stats[e]['TP'] += 1
    elif r['gt_answer']=='No'  and r['pred_answer']=='Yes': entity_stats[e]['FP'] += 1
    elif r['gt_answer']=='Yes' and r['pred_answer']=='No':  entity_stats[e]['FN'] += 1
    else:                                                   entity_stats[e]['TN'] += 1

entity_rows = []
for ent, s in entity_stats.items():
    p = s['TP'] / (s['TP']+s['FP']) if (s['TP']+s['FP']) > 0 else 0.0
    r = s['TP'] / (s['TP']+s['FN']) if (s['TP']+s['FN']) > 0 else 0.0
    f = (2*p*r)/(p+r) if (p+r) > 0 else 0.0
    entity_rows.append({
        'entity'   : ent,
        'total'    : s['total'],
        'TP': s['TP'], 'FP': s['FP'],
        'FN': s['FN'], 'TN': s['TN'],
        'precision': round(p, 4),
        'recall'   : round(r, 4),
        'f1'       : round(f, 4),
    })

entity_df = pd.DataFrame(entity_rows).sort_values('f1', ascending=False)

print(f"\nPer-Entity Results (top 20 by F1):")
print(entity_df.head(20).to_string(index=False))
print(f"\nBottom 10 entities by F1:")
print(entity_df.tail(10).to_string(index=False))

# == Save summary ==============================================
summary = {
    "model"     : "LLaVA-S (CheXpert finetuned)",
    "total_eval": len(valid),
    "skipped"   : skipped,
    "TP": TP, "FP": FP, "FN": FN, "TN": TN,
    "precision" : round(precision, 4),
    "recall"    : round(recall,    4),
    "f1"        : round(f1,        4),
    "accuracy"  : round(accuracy,  4),
    "per_entity": entity_rows,
}
with open(EVAL_SUMMARY_JSON, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\nSummary saved -> {EVAL_SUMMARY_JSON}")

## Result of this stage

The saved adapter is the LLaVA-S model. It understands chest radiography but is still a
single-image model and cannot use retrieved references, which is what the next stage addresses.